# MLLMs cloned-language GPT: Colab end-to-end

This notebook prepares TinyStories, trains or resumes the 12-layer model, evaluates original/clone behavior with BLiMP, runs activation patching, and displays all reports. Run cells from top to bottom. Data preparation and full training are long-running steps; checkpoints and reusable token assets are persisted to Google Drive.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPO_URL = "https://github.com/jiayi-ji01/mllms-colab.git"
PROJECT_DIR = Path("/content/mllms-colab")
DRIVE_ROOT = Path("/content/drive/MyDrive/mllms-colab")
RUN_DIR = DRIVE_ROOT / "runs/gpt12_tinystories_clone_colab"
ASSET_DIR = DRIVE_ROOT / "assets/tinystories_100m"
CONFIG_PATH = PROJECT_DIR / "configs/gpt12_tinystories_clone_colab.yaml"

TARGET_TRAIN_TOKENS = 100_000_000
RUN_TRAINING = True
RUN_BLIMP = True
RUN_PATCHING = True
PATCHING_EXAMPLES = 10
PATCHING_MAX_POSITIONS = 32

## 1. GPU and Google Drive

In [ ]:
import torch
from google.colab import drive

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone and install the project

In [ ]:
if PROJECT_DIR.exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-e", ".", "--no-deps"], check=True)
subprocess.run(["mllms", "--help"], check=True)

## 3. Prepare or restore TinyStories assets

The notebook restores tokenizer and binary token streams from Drive when available. Otherwise it downloads TinyStories, trains SentencePiece, tokenizes to at least 100M actual tokens, and backs up the reusable assets.

In [ ]:
tokenizer_model = PROJECT_DIR / "artifacts/tokenizer/tokenizer.model"
processed_dir = PROJECT_DIR / "data/processed"
required_bins = [processed_dir / f"{split}.bin" for split in ("train", "validation", "test")]
token_count_file = processed_dir / "token_counts.json"
local_ready = (
    tokenizer_model.is_file()
    and token_count_file.is_file()
    and all(path.is_file() for path in required_bins)
)
backup_ready = (ASSET_DIR / "tokenizer.model").is_file() and (ASSET_DIR / "token_counts.json").is_file() and all(
    (ASSET_DIR / path.name).is_file() for path in required_bins
)

if not local_ready and backup_ready:
    tokenizer_model.parent.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ASSET_DIR / "tokenizer.model", tokenizer_model)
    vocab_backup = ASSET_DIR / "tokenizer.vocab"
    if vocab_backup.is_file():
        shutil.copy2(vocab_backup, tokenizer_model.with_suffix(".vocab"))
    for path in required_bins:
        shutil.copy2(ASSET_DIR / path.name, path)
    shutil.copy2(ASSET_DIR / "token_counts.json", processed_dir / "token_counts.json")
    local_ready = True
    print("Restored tokenizer and token streams from Drive.")

if not local_ready:
    if not (PROJECT_DIR / "data/raw/manifest.json").is_file():
        subprocess.run(["mllms", "data", "prepare"], check=True)
    if not tokenizer_model.is_file():
        subprocess.run(["mllms", "tokenizer", "train"], check=True)
    subprocess.run(
        ["mllms", "data", "tokenize", "--target-train-tokens", str(TARGET_TRAIN_TOKENS)],
        check=True,
    )
    ASSET_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(tokenizer_model, ASSET_DIR / "tokenizer.model")
    tokenizer_vocab = tokenizer_model.with_suffix(".vocab")
    if tokenizer_vocab.is_file():
        shutil.copy2(tokenizer_vocab, ASSET_DIR / "tokenizer.vocab")
    for path in required_bins:
        shutil.copy2(path, ASSET_DIR / path.name)
    shutil.copy2(processed_dir / "token_counts.json", ASSET_DIR / "token_counts.json")
    print("Prepared assets and backed them up to Drive.")

token_counts = json.loads((processed_dir / "token_counts.json").read_text())
print(json.dumps(token_counts, indent=2))
for path in [tokenizer_model, *required_bins]:
    print(f"{path.relative_to(PROJECT_DIR)}: {path.stat().st_size / 2**20:.1f} MiB")

## 4. Train or resume the 12-layer model

If `final.pt` exists, training is skipped. If only `latest.pt` exists, training resumes with model, optimizer, scheduler, scaler, counters, and RNG state.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
final_checkpoint = RUN_DIR / "final.pt"
latest_checkpoint = RUN_DIR / "latest.pt"
best_checkpoint = RUN_DIR / "best.pt"

if RUN_TRAINING and not final_checkpoint.is_file():
    command = [
        "mllms", "train",
        "--config", str(CONFIG_PATH),
        "--output-dir", str(RUN_DIR),
    ]
    if latest_checkpoint.is_file():
        command.extend(["--resume", str(latest_checkpoint)])
        print("Resuming from:", latest_checkpoint)
    subprocess.run(command, check=True)
elif final_checkpoint.is_file():
    print("Training already complete:", final_checkpoint)
else:
    print("Training skipped by RUN_TRAINING=False")

assert best_checkpoint.is_file(), "best.pt is required for evaluation"

## 5. Pretraining curves and summary table

In [ ]:
from IPython.display import Image, display
import pandas as pd

subprocess.run(["mllms", "plot", "training", "--run-dir", str(RUN_DIR)], check=True)
display(Image(filename=str(RUN_DIR / "training_report.png")))
display(pd.read_csv(RUN_DIR / "training_summary.csv"))

## 6. BLiMP original/clone evaluation

In [ ]:
blimp_raw = PROJECT_DIR / "data/blimp/raw/agreement.jsonl"
blimp_processed = PROJECT_DIR / "data/blimp/processed/agreement.jsonl"
blimp_dir = RUN_DIR / "blimp"
blimp_summary = blimp_dir / "agreement_summary.json"

if RUN_BLIMP:
    if not blimp_raw.is_file():
        subprocess.run(["mllms", "blimp", "download"], check=True)
    if not blimp_processed.is_file():
        subprocess.run(
            ["mllms", "blimp", "prepare", "--checkpoint", str(best_checkpoint)],
            check=True,
        )
    if not blimp_summary.is_file():
        subprocess.run(
            [
                "mllms", "blimp", "evaluate",
                "--checkpoint", str(best_checkpoint),
                "--output-dir", str(blimp_dir),
                "--device", "cuda",
            ],
            check=True,
        )

if blimp_summary.is_file():
    subprocess.run(["mllms", "plot", "blimp", "--results-dir", str(blimp_dir)], check=True)
    display(Image(filename=str(blimp_dir / "blimp_report.png")))
    display(pd.read_csv(blimp_dir / "blimp_summary.csv"))

## 7. Original and clone activation patching

The first command builds a small bank of grammatical singular/plural prompt pairs and retains only pairs aligned by the trained SentencePiece tokenizer. Patching is run separately for original and clone token spaces.

In [ ]:
sva_data = PROJECT_DIR / "data/sva_pairs.jsonl"
if RUN_PATCHING:
    subprocess.run(
        [
            "mllms", "analyze", "prepare-sva",
            "--output", str(sva_data),
            "--max-examples", str(PATCHING_EXAMPLES),
        ],
        check=True,
    )

    for language in ("original", "clone"):
        output_dir = RUN_DIR / f"activation_patching_{language}"
        if not (output_dir / "metadata.json").is_file():
            subprocess.run(
                [
                    "mllms", "analyze", "activation-patching",
                    "--checkpoint", str(best_checkpoint),
                    "--data", str(sva_data),
                    "--output-dir", str(output_dir),
                    "--device", "cuda",
                    "--language", language,
                    "--max-examples", str(PATCHING_EXAMPLES),
                    "--max-positions", str(PATCHING_MAX_POSITIONS),
                ],
                check=True,
            )
        subprocess.run(
            ["mllms", "plot", "patching", "--results-dir", str(output_dir)],
            check=True,
        )

In [ ]:
for language in ("original", "clone"):
    output_dir = RUN_DIR / f"activation_patching_{language}"
    report = output_dir / "patching_report.png"
    table = output_dir / "patching_top_sites.csv"
    if report.is_file():
        print(f"{language.upper()} PATCHING")
        display(Image(filename=str(report)))
        display(pd.read_csv(table).head(20))

## 8. Final artifact inventory

In [ ]:
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(RUN_DIR)}  ({path.stat().st_size / 2**20:.2f} MiB)")